In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Cerebrasensecloud')
KAGGLE_DRIVE_ROOT = DRIVE_ROOT / 'kaggle_alz_upload_bundle'
RUNTIME_ROOT = DRIVE_ROOT / 'backend_runtime'
LOCAL_KAGGLE_STAGE = Path('/content/kaggle_stage/kaggle_alz_upload_bundle')
RUN_NAME = 'kaggle_colab_baseline_v1'

if not KAGGLE_DRIVE_ROOT.exists():
    raise FileNotFoundError(
        f'Kaggle upload bundle not found: {KAGGLE_DRIVE_ROOT}. '
        'Upload the kaggle_alz_upload_bundle folder to Drive and point KAGGLE_DRIVE_ROOT at it.'
    )

RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
print('drive_root =', DRIVE_ROOT)
print('kaggle_drive_root =', KAGGLE_DRIVE_ROOT)
print('runtime_root =', RUNTIME_ROOT)
print('local_kaggle_stage =', LOCAL_KAGGLE_STAGE)
print('run_name =', RUN_NAME)


In [ ]:
import shutil
import subprocess
from pathlib import Path

REPO_ROOT = Path('/content/Cerebrasense-')
BACKEND_ROOT = REPO_ROOT / 'alz_backend'

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(
    ['git', 'clone', 'https://github.com/Billrichard209/Cerebrasense-.git'],
    cwd='/content',
    check=True,
)

if not BACKEND_ROOT.exists():
    raise FileNotFoundError(f'Expected backend after clone: {BACKEND_ROOT}')

print('repo_root =', REPO_ROOT)
print('backend_root =', BACKEND_ROOT)


In [ ]:
import subprocess
import sys

requirements = BACKEND_ROOT / 'requirements-colab.txt'
if not requirements.exists():
    raise FileNotFoundError(f'Missing requirements file: {requirements}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)], check=True)
print('dependencies ready =', requirements)


In [ ]:
EPOCHS = 12
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 1
NUM_WORKERS = 0
CACHE_RATE = 0.0
IMAGE_SIZE_2D = (224, 224)
IMAGE_SIZE_3D = (128, 128, 128)
SEED = 42
SPLIT_RANDOM_STATE = 42
TRAIN_FRACTION = 0.7
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15
DROPOUT_PROB = 0.0
DRY_RUN = False
DETERMINISTIC = True
EARLY_STOPPING_MONITOR = 'val_macro_f1'
EARLY_STOPPING_MODE = 'max'
EARLY_STOPPING_PATIENCE = 4
EARLY_STOPPING_MIN_DELTA = 0.0
STAGE_KAGGLE_TO_LOCAL = True
FORCE_RESTAGE = False
RESUME_IF_AVAILABLE = True
MAX_TRAIN_BATCHES = None
MAX_VAL_BATCHES = None
MAX_TEST_BATCHES = None

print('epochs =', EPOCHS)
print('batch_size =', BATCH_SIZE)
print('gradient_accumulation_steps =', GRADIENT_ACCUMULATION_STEPS)
print('split_random_state =', SPLIT_RANDOM_STATE)
print('stage_kaggle_to_local =', STAGE_KAGGLE_TO_LOCAL)
print('resume_if_available =', RESUME_IF_AVAILABLE)


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    'scripts/train_kaggle_colab.py',
    '--project-root', str(BACKEND_ROOT),
    '--runtime-root', str(RUNTIME_ROOT),
    '--kaggle-source-dir', str(KAGGLE_DRIVE_ROOT),
    '--run-name', RUN_NAME,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--gradient-accumulation-steps', str(GRADIENT_ACCUMULATION_STEPS),
    '--num-workers', str(NUM_WORKERS),
    '--cache-rate', str(CACHE_RATE),
    '--image-size-2d', *(str(value) for value in IMAGE_SIZE_2D),
    '--image-size-3d', *(str(value) for value in IMAGE_SIZE_3D),
    '--seed', str(SEED),
    '--split-random-state', str(SPLIT_RANDOM_STATE),
    '--train-fraction', str(TRAIN_FRACTION),
    '--val-fraction', str(VAL_FRACTION),
    '--test-fraction', str(TEST_FRACTION),
    '--dropout-prob', str(DROPOUT_PROB),
    '--early-stopping-monitor', EARLY_STOPPING_MONITOR,
    '--early-stopping-mode', EARLY_STOPPING_MODE,
    '--early-stopping-patience', str(EARLY_STOPPING_PATIENCE),
    '--early-stopping-min-delta', str(EARLY_STOPPING_MIN_DELTA),
    '--local-kaggle-root', str(LOCAL_KAGGLE_STAGE),
]

cmd.append('--dry-run' if DRY_RUN else '--no-dry-run')
cmd.append('--deterministic' if DETERMINISTIC else '--no-deterministic')
cmd.append('--stage-kaggle-to-local' if STAGE_KAGGLE_TO_LOCAL else '--no-stage-kaggle-to-local')
cmd.append('--resume-if-available' if RESUME_IF_AVAILABLE else '--no-resume-if-available')
if FORCE_RESTAGE:
    cmd.append('--force-restage')
if MAX_TRAIN_BATCHES is not None:
    cmd.extend(['--max-train-batches', str(MAX_TRAIN_BATCHES)])
if MAX_VAL_BATCHES is not None:
    cmd.extend(['--max-val-batches', str(MAX_VAL_BATCHES)])
if MAX_TEST_BATCHES is not None:
    cmd.extend(['--max-test-batches', str(MAX_TEST_BATCHES)])

print('RUNNING:', ' '.join(cmd))
subprocess.run(cmd, cwd=BACKEND_ROOT, check=True)


In [ ]:
import json
from pathlib import Path

RUNTIME_DATA_ROOT = RUNTIME_ROOT / 'data'
RUNTIME_OUTPUTS_ROOT = RUNTIME_ROOT / 'outputs'
RUN_ROOT = RUNTIME_OUTPUTS_ROOT / 'runs' / 'kaggle' / RUN_NAME
SUMMARY_PATH = RUN_ROOT / 'reports' / 'colab_run_summary.json'

print('runtime_data_root =', RUNTIME_DATA_ROOT)
print('runtime_outputs_root =', RUNTIME_OUTPUTS_ROOT)
print('run_root =', RUN_ROOT)
print('summary_path =', SUMMARY_PATH)

if SUMMARY_PATH.exists():
    print(json.dumps(json.loads(SUMMARY_PATH.read_text(encoding='utf-8')), indent=2))
else:
    print('summary file not found yet')


In [ ]:
# Optional: run one sample inference from the saved Kaggle checkpoint.
import json
import os
import pandas as pd
from pathlib import Path
import sys

RUNTIME_DATA_ROOT = RUNTIME_ROOT / 'data'
RUNTIME_OUTPUTS_ROOT = RUNTIME_ROOT / 'outputs'
summary_path = RUNTIME_OUTPUTS_ROOT / 'runs' / 'kaggle' / RUN_NAME / 'reports' / 'colab_run_summary.json'

if not summary_path.exists():
    raise FileNotFoundError(f'Summary missing: {summary_path}')

summary = json.loads(summary_path.read_text(encoding='utf-8'))
resolved_config_path = Path(summary['resolved_config_path'])
if not resolved_config_path.exists():
    raise FileNotFoundError(f'Resolved config missing: {resolved_config_path}')

os.environ['ALZ_DATA_ROOT'] = str(RUNTIME_DATA_ROOT)
os.environ['ALZ_OUTPUTS_ROOT'] = str(RUNTIME_OUTPUTS_ROOT)
os.environ['ALZ_KAGGLE_SOURCE_DIR'] = str(LOCAL_KAGGLE_STAGE if LOCAL_KAGGLE_STAGE.exists() else KAGGLE_DRIVE_ROOT)
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

import torch
from src.evaluation.kaggle_run import load_kaggle_checkpoint
from src.inference.predict_kaggle import predict_kaggle_image
from src.models.kaggle_model import KaggleMonaiModelConfig, build_kaggle_monai_network

resolved_config = json.loads(resolved_config_path.read_text(encoding='utf-8'))
dataset_type = resolved_config['dataset_type']
class_names = tuple(resolved_config['class_names'])
checkpoint_path = Path(summary['best_checkpoint'] or summary['last_checkpoint'])
if not checkpoint_path.exists():
    raise FileNotFoundError(f'Checkpoint missing: {checkpoint_path}')

manifest_path = Path(summary['val_manifest_path'])
if not manifest_path.exists():
    manifest_path = Path(summary['train_manifest_path'])
manifest = pd.read_csv(manifest_path)
sample_image = Path(manifest.iloc[0]['image'])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
checkpoint = load_kaggle_checkpoint(checkpoint_path, device=device)
model = build_kaggle_monai_network(
    KaggleMonaiModelConfig(
        dataset_type=dataset_type,
        in_channels=1,
        out_channels=len(class_names),
        dropout_prob=float(resolved_config['training'].get('dropout_prob', 0.0)),
    )
)
model.load_state_dict(checkpoint.model_state_dict)

prediction = predict_kaggle_image(
    sample_image,
    model=model,
    dataset_type=dataset_type,
    class_names=class_names,
    device=device,
)

print('sample_image =', sample_image)
print('predicted_label =', prediction.label)
print('predicted_index =', prediction.predicted_index)
print('confidence =', prediction.confidence)
print('probabilities =', prediction.probabilities)
